In [ ]:
import numpy as np, itertools, sys
def tz(x): return (x&-x).bit_length()-1
def o1(i,j):
    if i==0 or j==0: return 1
    if i==j: return -1
    t=tz(i^j); ti=tz(i); tj=tz(j); k=max(t,ti,tj)+1
    if t>=ti and t>=tj: s=2*((i>>t)&1)-1
    elif ti>t and ti>=tj:
        b=(j>>ti)&1; jl=j&((1<<ti)-1); s=(-1)**b*(1 if jl==0 else -1)
    else: s=1-2*((i>>tj)&1)
    if bin((i|j)>>k).count("1")%2: s=-s
    return s

# 1. split relation: final-doubling split sign = standard sign * (-1)^(a*b) (top bits), check vs block rules of Thm 4.1
def split(i,j,m):
    h=1<<(m-1)
    if i>=h and j>=h:
        il,jl=i-h,j-h
        if il==jl: return 1
        if il==0: return -1
        if jl==0: return 1
        return o1(il,jl)
    return o1(i,j)
bad=0
for m in range(1,9):
    N=1<<m
    for i in range(N):
        for j in range(N):
            a=(i>>(m-1))&1;b=(j>>(m-1))&1
            if split(i,j,m)!=o1(i,j)*(-1)**(a*b): bad+=1
print("split vs std*(-1)^(ab) mismatches:",bad)

# 2. Lemma 8.13
bad=0
for r in range(1,9):
    for k in range(1,2000):
        if o1(2**r*k,2**r*k+1)!=(-1)**(bin(k).count('1')+1): bad+=1
print("Lemma 8.13 mismatches:",bad)

# 3. Statement 8.6 (2x2 block product -1, top-left never unique minority)
for n in range(2,10):
    N=1<<n
    S=np.array([[o1(i,j) for j in range(N)] for i in range(N)])
    A=S[0::2,0::2];B=S[0::2,1::2];C=S[1::2,0::2];D=S[1::2,1::2]
    prod=A*B*C*D
    cond1=(prod==-1).all()
    # unique minority top-left: A differs from B,C,D which all equal
    mino=((B==C)&(C==D)&(A!=B)).any()
    print("n",n,"prod=-1 all:",cond1,"| TL unique minority exists:",mino)

# 4. alphabet ladder and features, n=10
n=10;N=1<<n
S=np.array([[o1(i,j) for j in range(N)] for i in range(N)])
M=(S==-1).astype(int)+1
levels=[M]
sizes=[len(np.unique(M))]
cur=M
while cur.shape[0]>1:
    h=cur.shape[0]//2
    T=np.stack([cur[0::2,0::2],cur[0::2,1::2],cur[1::2,0::2],cur[1::2,1::2]],-1).reshape(h*h,4)
    u,inv=np.unique(T,axis=0,return_inverse=True)
    cur=(inv.reshape(h,h)+1)
    levels.append(cur); sizes.append(len(u))
print("ladder n=10:",sizes)

def feat(a,b):
    def p(x): return 0 if x==0 else bin((a|b)>>tz(x)).count("1")%2
    def q(x,y): return 0 if x==0 else (y>>tz(x))&1
    return (int(b==0),int(a==b),p(a),p(b),p(a^b),q(a,b),q(b,a),q(a^b,a))
mp={}
conf=0
for k in (3,4,5,6):
    L=levels[k];h=L.shape[0]
    for a in range(h):
        for b in range(h):
            f=feat(a,b)
            s=int(L[a,b])
            if f in mp and mp[f]!=s: conf+=1
            mp.setdefault(f,s)
print("distinct feature vectors:",len(mp),", symbols:",len(set(mp.values())),", conflicts across levels 3-6:",conf)
for m in mp:
    print(m)

split vs std*(-1)^(ab) mismatches: 0
Lemma 8.13 mismatches: 0
n 2 prod=-1 all: True | TL unique minority exists: False
n 3 prod=-1 all: True | TL unique minority exists: False
n 4 prod=-1 all: True | TL unique minority exists: False
n 5 prod=-1 all: True | TL unique minority exists: False
n 6 prod=-1 all: True | TL unique minority exists: False
n 7 prod=-1 all: True | TL unique minority exists: False
n 8 prod=-1 all: True | TL unique minority exists: False
n 9 prod=-1 all: True | TL unique minority exists: False
ladder n=10: [2, 6, 12, 12, 12, 12, 12, 12, 9, 4, 1]
distinct feature vectors: 29 , symbols: 12 , conflicts across levels 3-5: 0
(1, 1, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 1, 1, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0)
(1, 0, 1, 0, 1, 0, 0, 1)
(0, 1, 1, 1, 0, 1, 1, 0)
(0, 0, 0, 1, 0, 0, 0, 1)
(0, 0, 0, 0, 1, 1, 1, 0)
(0, 0, 1, 0, 1, 0, 0, 1)
(0, 0, 1, 1, 0, 1, 1, 0)
(0, 0, 1, 0, 0, 0, 0, 0)
(0, 0, 1, 0, 0, 1, 0, 0)
(0, 0, 0, 1, 1, 1, 0, 0)
(1, 0, 0, 0, 0, 0, 0, 1)
(0, 0, 0, 0, 1, 1, 1, 1)
(0,

In [7]:

def tz(x): return (x&-x).bit_length()-1
def pop(x): return x.bit_count()
def o1(i,j):
    if i==0 or j==0: return 1
    if i==j: return -1
    t=tz(i^j); ti=tz(i); tj=tz(j); kk=max(t,ti,tj)+1
    if t>=ti and t>=tj: s=2*((i>>t)&1)-1
    elif ti>t and ti>=tj:
        bb=(j>>ti)&1; jl=j&((1<<ti)-1)
        s=(-1)**bb*(1 if jl==0 else -1)
    else: s=1-2*((i>>tj)&1)
    return -s if pop((i|j)>>kk)%2 else s
def ff(a,b):
    def p(x): return 0 if x==0 else pop((a|b)>>tz(x))%2
    def q(x,y): return 0 if x==0 else (y>>tz(x))&1
    return (int(b==0),int(a==b),p(a),p(b),p(a^b),q(a,b),q(b,a),q(a^b,a))
def sig(a,b):
    b0,d0,pa,pb,px,qab,qba,qxa=ff(a,b)
    pi=pa if (d0 or b0 or (qab==0 and qxa==1)) else pb
    rho=(-1)**(qab+pa)
    kap=1 if b0 else (-1)**(qba+pb+1)
    delt=-1 if d0 else (-1)**(qxa+px)
    eps=(-1)**pi
    if b0 or a==0: omg=1
    elif d0: omg=-1
    elif qab==qba==1: omg=(-1)**(qxa+px)
    elif qab==0 and qxa==1: omg=(-1)**(qba+pb+1)
    else: omg=(-1)**(qab+pa)
    return rho,kap,delt,eps,omg
def pred(k,a,b,r,c):
    rho,kap,delt,eps,omg=sig(a,b)
    if r==c==0: return omg
    if r==0: return rho
    if c==0: return kap
    if r==c: return delt
    return eps*o1(r,c)
for k in range(1,7):
    N=1<<k; bad=None
    for a in range(8):
      for b in range(8):
       for r in range(N):
        for c in range(N):
         actual=o1(N*a+r,N*b+c)
         expected=pred(k,a,b,r,c)
         if actual!=expected:
          bad=(a,b,r,c,actual,expected); break
        if bad: break
       if bad: break
      if bad: break
    print(k,bad)


for m in [8,4,2,1]:
    vals={sig(a,b) for a in range(m) for b in range(m)}
    print(m,len(vals),sorted(vals))


def tz(x): return (x&-x).bit_length()-1
def pop(x): return x.bit_count()
def ff(a,b):
    def p(x): return 0 if x==0 else pop((a|b)>>tz(x))%2
    def q(x,y): return 0 if x==0 else (y>>tz(x))&1
    return (int(b==0),int(a==b),p(a),p(b),p(a^b),q(a,b),q(b,a),q(a^b,a))
def sig(a,b):
    b0,d0,pa,pb,px,qab,qba,qxa=ff(a,b)
    pi=pa if (d0 or b0 or (qab==0 and qxa==1)) else pb
    rho=(-1)**(qab+pa)
    kap=1 if b0 else (-1)**(qba+pb+1)
    delt=-1 if d0 else (-1)**(qxa+px)
    eps=(-1)**pi
    if b0 or a==0: omg=1
    elif d0: omg=-1
    elif qab==qba==1: omg=(-1)**(qxa+px)
    elif qab==0 and qxa==1: omg=(-1)**(qba+pb+1)
    else: omg=(-1)**(qab+pa)
    return rho,kap,delt,eps,omg
for m in (8,4,2):
    reps={}
    for a in range(m):
      for b in range(m):
       reps.setdefault(sig(a,b),(a,b))
    print("m",m, "count",len(reps))
    print(list(reps.items()))



1 None
2 None
3 None
4 None
5 None
6 None
8 12 [(-1, -1, 1, -1, -1), (-1, -1, 1, 1, -1), (-1, 1, -1, -1, -1), (-1, 1, -1, 1, -1), (-1, 1, 1, -1, 1), (-1, 1, 1, 1, 1), (1, -1, -1, -1, -1), (1, -1, -1, 1, -1), (1, -1, 1, -1, 1), (1, -1, 1, 1, 1), (1, 1, -1, -1, 1), (1, 1, -1, 1, 1)]
4 9 [(-1, -1, 1, 1, -1), (-1, 1, -1, 1, -1), (-1, 1, 1, -1, 1), (-1, 1, 1, 1, 1), (1, -1, -1, -1, -1), (1, -1, -1, 1, -1), (1, -1, 1, 1, 1), (1, 1, -1, -1, 1), (1, 1, -1, 1, 1)]
2 4 [(-1, 1, 1, -1, 1), (1, -1, -1, -1, -1), (1, 1, -1, -1, 1), (1, 1, -1, 1, 1)]
1 1 [(1, 1, -1, 1, 1)]
m 8 count 12
[((1, 1, -1, 1, 1), (0, 0)), ((1, 1, -1, -1, 1), (0, 1)), ((1, -1, 1, 1, 1), (0, 3)), ((-1, 1, 1, -1, 1), (1, 0)), ((1, -1, -1, -1, -1), (1, 1)), ((-1, 1, -1, 1, -1), (1, 3)), ((-1, -1, 1, -1, -1), (1, 6)), ((1, -1, 1, -1, 1), (1, 7)), ((-1, -1, 1, 1, -1), (2, 1)), ((-1, 1, -1, -1, -1), (2, 7)), ((-1, 1, 1, 1, 1), (3, 1)), ((1, -1, -1, 1, -1), (3, 2))]
m 4 count 9
[((1, 1, -1, 1, 1), (0, 0)), ((1, 1, -1, -1, 1), (0, 1)

In [8]:
for k in range(1,7):
    N=1<<k; bad=None
    for a in range(16):
      for b in range(16):
        for r in range(N):
          for c in range(N):
            if o1(N*a+r,N*b+c)!=pred(k,a,b,r,c): bad=(a,b,r,c); break
          if bad: break
        if bad: break
      if bad: break
    print("k",k,bad)

k 1 None
k 2 None
k 3 None
k 4 None
k 5 None
k 6 None


In [9]:
from collections import defaultdict
S16={sig(a,b) for a in range(16) for b in range(16)}
S32={sig(a,b) for a in range(32) for b in range(32)}
print("m=16:",len(S16),"  m=32:",len(S32))       # expect 12, 12

g=defaultdict(set)
for rho,kap,delt,eps,omg in S16: g[(rho,kap,delt,omg)].add(eps)
print("level-1 patterns:",len(g))                # expect 6
print("every pattern has both parities:",all(v=={-1,1} for v in g.values()))
for (rho,kap,delt,omg) in sorted(g):
    exc=[n for n,s in zip("bcd",(rho,kap,delt)) if s!=omg]
    print((rho,kap,delt,omg),"prod",rho*kap*delt*omg,"exc",exc)

m=16: 12   m=32: 12
level-1 patterns: 6
every pattern has both parities: True
(-1, -1, 1, -1) prod -1 exc ['d']
(-1, 1, -1, -1) prod -1 exc ['c']
(-1, 1, 1, 1) prod -1 exc ['b']
(1, -1, -1, -1) prod -1 exc ['b']
(1, -1, 1, 1) prod -1 exc ['c']
(1, 1, -1, 1) prod -1 exc ['d']
